# Factor Model: Step 5 - Cross-Sectional Regression

## Paleologo's Framework: Step 5 of 6

**Objective**: Estimate factor returns by regressing stock returns against factor loadings daily using **Weighted Least Squares (WLS)**.

### The Cross-Sectional Regression Model (Equation 6.1):

$$\mathbf{r}_t = \mathbf{B}_t \mathbf{f}_t + \boldsymbol{\epsilon}_t, \quad t \in \mathbb{N}$$

Where:
- $\mathbf{r}_t \in \mathbb{R}^{n}$: Vector of returns for $n$ stocks at time $t$ (from Step 3 - winsorized)
- $\mathbf{B}_t \in \mathbb{R}^{n \times m}$: Matrix of factor loadings ($n$ stocks × $m$ factors from Step 4)
- $\mathbf{f}_t \in \mathbb{R}^{m}$: Vector of factor returns at time $t$ (**TO BE ESTIMATED**)
- $\boldsymbol{\epsilon}_t \in \mathbb{R}^{n}$: Vector of idiosyncratic returns (residuals)

### Goal:
For each trading day $t$, estimate $\mathbf{f}_t$ (factor returns) by solving:

$$\hat{\mathbf{f}}_t = \underset{\mathbf{f}_t}{\arg\min} \, L(\mathbf{r}_t - \mathbf{B}_t \mathbf{f}_t)$$

### Loss Function (Equation 6.2) - Weighted Least Squares:

$$L(\mathbf{r}_t - \mathbf{B}_t \mathbf{f}_t) := (\mathbf{r}_t - \mathbf{B}_t \mathbf{f}_t)^T \mathbf{W}_t (\mathbf{r}_t - \mathbf{B}_t \mathbf{f}_t)$$

Where:
- $\mathbf{W}_t = \text{diag}(w_1, w_2, ..., w_n)$: Diagonal weight matrix for stocks at time $t$
- $w_i = \frac{1}{\sigma_i^2}$: Weight inversely proportional to residual variance (heteroskedasticity adjustment)

### Solution (Equation 6.3) - Weighted Least Squares:

$$\hat{\mathbf{f}}_t = (\mathbf{B}_t^T \mathbf{W}_t \mathbf{B}_t)^{-1} \mathbf{B}_t^T \mathbf{W}_t \mathbf{r}_t$$

### Weight Estimation:

**Two-Pass Approach**:
1. **Pass 1 (Initialization)**: Run unweighted regression to get initial residuals
2. **Calculate Weights**: Estimate residual variance for each stock using rolling window
3. **Pass 2 (WLS)**: Re-run regression with estimated weights

**Weight Formula**:
$$w_i = \frac{1}{\hat{\sigma}_i^2 + \epsilon}$$

Where:
- $\hat{\sigma}_i^2$: Rolling standard deviation of stock $i$'s residuals (e.g., 60-day window)
- $\epsilon$: Small constant to prevent division by zero

---

## Implementation Strategy:

1. **Run initial unweighted regressions** to get baseline residuals
2. **Estimate residual volatilities** using rolling window (60 days)
3. **Calculate weights**: $w_i = 1 / \sigma_i^2$
4. **Run WLS regressions** with weights for all trading days
5. **Analyze factor returns** and diagnostics

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from tqdm import tqdm
from sklearn.linear_model import LinearRegression
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: '%.6f' % x)

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("Libraries imported successfully")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

## 5.1 Load Data from Previous Steps

In [ ]:
print("="*80)
print("LOADING DATA")
print("="*80)

# Load winsorized returns from Step 3
print("\nLoading winsorized returns from Step 3...")
df_winsorized = pd.read_parquet('russell2000_winsorized_step3.parquet')

print(f"✓ Winsorized data loaded")
print(f"  Shape: {df_winsorized.shape}")
print(f"  Index: {df_winsorized.index.names}")
print(f"  Columns: {df_winsorized.columns.tolist()}")

# Load factor loadings from Step 4
print("\nLoading factor loadings from Step 4...")
df_loadings = pd.read_parquet('russell2000_factor_loadings_step4.parquet')

print(f"✓ Factor loadings loaded")
print(f"  Shape: {df_loadings.shape}")
print(f"  Factors: {len(df_loadings.columns)}")
print(f"  Sample factors: {df_loadings.columns[:5].tolist()}")

## 5.2 Prepare Data for Regression

We need to align returns and loadings on the same (date, symbol) index.

In [ ]:
print("="*80)
print("DATA PREPARATION")
print("="*80)

# Extract winsorized returns
# r_t: returns vector for each date
print("\nExtracting winsorized returns...")
returns = df_winsorized['return_winsorized'].copy()

print(f"✓ Returns extracted")
print(f"  Shape: {returns.shape}")
print(f"  Non-null: {returns.notna().sum():,}")
print(f"  Date range: {returns.index.get_level_values(0).min()} to {returns.index.get_level_values(0).max()}")

# Align returns and loadings on common index
print("\nAligning returns and loadings...")
common_index = returns.index.intersection(df_loadings.index)

returns_aligned = returns.loc[common_index]
loadings_aligned = df_loadings.loc[common_index]

print(f"✓ Data aligned")
print(f"  Common observations: {len(common_index):,}")
print(f"  Unique dates: {common_index.get_level_values(0).nunique():,}")
print(f"  Unique symbols: {common_index.get_level_values(1).nunique():,}")

# Check for missing values
print(f"\nMissing value check:")
print(f"  Returns missing: {returns_aligned.isna().sum()} ({returns_aligned.isna().sum()/len(returns_aligned)*100:.2f}%)")
print(f"  Loadings missing: {loadings_aligned.isna().sum().sum()} ({loadings_aligned.isna().sum().sum()/loadings_aligned.size*100:.2f}%)")

# Drop rows with any missing values (complete case analysis)
valid_mask = returns_aligned.notna() & loadings_aligned.notna().all(axis=1)
returns_clean = returns_aligned[valid_mask]
loadings_clean = loadings_aligned[valid_mask]

print(f"\n✓ Clean data prepared (complete cases only)")
print(f"  Observations: {len(returns_clean):,}")
print(f"  Dates: {returns_clean.index.get_level_values(0).nunique():,}")
print(f"  Stocks per day (avg): {len(returns_clean) / returns_clean.index.get_level_values(0).nunique():.0f}")

## 5.3 Pass 1: Run Initial Unweighted Regressions

First pass to get initial residuals for weight estimation.

### Formula:
$$\hat{\mathbf{f}}_t^{(0)} = (\mathbf{B}_t^T \mathbf{B}_t)^{-1} \mathbf{B}_t^T \mathbf{r}_t$$

In [ ]:
print("="*80)
print("PASS 1: INITIAL UNWEIGHTED REGRESSIONS")
print("="*80)
print("\nRunning initial regressions to estimate residuals...")
print("Model: r_t = B_t @ f_t + epsilon_t (unweighted)\n")

# Get unique dates
dates = returns_clean.index.get_level_values(0).unique().sort_values()

print(f"Total dates to process: {len(dates):,}")
print(f"Date range: {dates[0]} to {dates[-1]}")
print(f"Factors: {len(loadings_clean.columns)}\n")

# Storage for Pass 1 results
residuals_pass1 = []  # Initial residuals for weight estimation

# Run initial unweighted regression for each date
for date in tqdm(dates, desc="Pass 1 - Initial regressions"):
    try:
        # Extract data for this date
        r_t = returns_clean.xs(date, level=0)
        B_t = loadings_clean.xs(date, level=0)
        
        # Skip if insufficient data
        if len(r_t) < len(loadings_clean.columns):
            continue
        
        # Run unweighted OLS regression
        model = LinearRegression(fit_intercept=False)
        model.fit(B_t.values, r_t.values)
        
        # Calculate residuals
        r_t_fitted = model.predict(B_t.values)
        epsilon_t = r_t.values - r_t_fitted
        
        # Store residuals with (date, symbol) index
        for symbol, resid in zip(r_t.index, epsilon_t):
            residuals_pass1.append({
                'date': date,
                'symbol': symbol,
                'residual': resid
            })
        
    except Exception as e:
        continue

# Convert to DataFrame
df_residuals_pass1 = pd.DataFrame(residuals_pass1)
df_residuals_pass1 = df_residuals_pass1.set_index(['date', 'symbol'])['residual']

print(f"\n✓ Pass 1 complete")
print(f"  Residuals calculated: {len(df_residuals_pass1):,}")
print(f"  Dates: {df_residuals_pass1.index.get_level_values(0).nunique():,}")
print(f"  Unique symbols: {df_residuals_pass1.index.get_level_values(1).nunique():,}")

## 5.4 Organize Results into DataFrames

In [ ]:
print("="*80)
print("PASS 2: WEIGHTED LEAST SQUARES (WLS) REGRESSIONS")
print("="*80)
print("\nRunning WLS regressions with estimated weights...")
print("Model: r_t = B_t @ f_t + epsilon_t (weighted)")
print("Formula: f_t = (B_t^T W_t B_t)^(-1) B_t^T W_t r_t\n")

# Storage for WLS results
factor_returns_list = []  # Daily factor returns (f_t)
residuals_list = []        # Daily residuals (epsilon_t)
r_squared_list = []        # Daily R-squared
n_stocks_list = []         # Number of stocks per day

# Get dates where we have weights available
dates_with_weights = weights_stacked.index.get_level_values(0).unique().sort_values()

print(f"Dates with weights available: {len(dates_with_weights):,}")
print(f"Date range: {dates_with_weights[0]} to {dates_with_weights[-1]}\n")

# Run WLS regression for each date
for date in tqdm(dates_with_weights, desc="Pass 2 - WLS regressions"):
    try:
        # Extract data for this date
        r_t = returns_clean.xs(date, level=0)
        B_t = loadings_clean.xs(date, level=0)
        
        # Get weights for this date
        try:
            w_t = weights_stacked.xs(date, level=0)
        except KeyError:
            # No weights for this date, skip
            continue
        
        # Align weights with returns/loadings
        common_symbols = r_t.index.intersection(B_t.index).intersection(w_t.index)
        if len(common_symbols) < len(loadings_clean.columns):
            # Not enough stocks
            continue
        
        r_t_aligned = r_t.loc[common_symbols]
        B_t_aligned = B_t.loc[common_symbols]
        w_t_aligned = w_t.loc[common_symbols]
        
        # Convert to numpy arrays
        r = r_t_aligned.values
        B = B_t_aligned.values
        w = w_t_aligned.values
        
        # Create diagonal weight matrix W
        W = np.diag(w)
        
        # WLS formula: f_t = (B^T W B)^(-1) B^T W r
        BtWB = B.T @ W @ B
        BtWr = B.T @ W @ r
        
        # Solve for factor returns
        f_t = np.linalg.solve(BtWB, BtWr)
        
        # Calculate fitted values: B @ f_t
        r_t_fitted = B @ f_t
        
        # Calculate residuals: epsilon_t = r - B @ f_t
        epsilon_t = r - r_t_fitted
        
        # Calculate R-squared
        ss_total = np.sum((r - r.mean()) ** 2)
        ss_residual = np.sum(epsilon_t ** 2)
        r_squared = 1 - (ss_residual / ss_total) if ss_total > 0 else 0
        
        # Store results
        factor_returns_list.append(pd.Series(f_t, index=loadings_clean.columns, name=date))
        residuals_list.append(pd.Series(epsilon_t, index=common_symbols, name=date))
        r_squared_list.append(r_squared)
        n_stocks_list.append(len(r))
        
    except Exception as e:
        print(f"\n⚠ Error on {date}: {e}")
        continue

print(f"\n✓ WLS regressions complete")
print(f"  Successful: {len(factor_returns_list):,} days")
print(f"  Failed: {len(dates_with_weights) - len(factor_returns_list):,} days")

## 5.5 Pass 2: Run Weighted Least Squares (WLS) Regressions

Use the estimated weights to run WLS regressions for all days.

### Formula (Equation 6.3):
$$\hat{\mathbf{f}}_t = (\mathbf{B}_t^T \mathbf{W}_t \mathbf{B}_t)^{-1} \mathbf{B}_t^T \mathbf{W}_t \mathbf{r}_t$$

In [ ]:
print("="*80)
print("CALCULATING WEIGHTS FROM RESIDUAL VOLATILITY")
print("="*80)

# Parameters
ROLLING_WINDOW = 60  # Days for rolling variance estimation
MIN_PERIODS = 20      # Minimum observations required
EPSILON = 1e-8        # Small constant to prevent division by zero

print(f"\nParameters:")
print(f"  Rolling window: {ROLLING_WINDOW} days")
print(f"  Minimum periods: {MIN_PERIODS} days")
print(f"  Epsilon (regularization): {EPSILON}")

# Calculate rolling std of residuals for each stock
print(f"\nCalculating rolling standard deviation of residuals by stock...")

# Unstack to get (date, symbol) panel
residuals_panel = df_residuals_pass1.unstack()

# Calculate rolling std for each symbol (column)
rolling_std = residuals_panel.rolling(window=ROLLING_WINDOW, min_periods=MIN_PERIODS).std()

# Square to get variance
rolling_var = rolling_std ** 2

# Calculate weights: w = 1 / (variance + epsilon)
weights = 1.0 / (rolling_var + EPSILON)

# Stack back to (date, symbol) format
weights_stacked = weights.stack()
weights_stacked.name = 'weight'

print(f"✓ Weights calculated")
print(f"  Total weights: {len(weights_stacked):,}")
print(f"  Date range: {weights_stacked.index.get_level_values(0).min()} to {weights_stacked.index.get_level_values(0).max()}")

# Statistics
print(f"\nWeight statistics:")
print(f"  Mean: {weights_stacked.mean():.2f}")
print(f"  Median: {weights_stacked.median():.2f}")
print(f"  Std: {weights_stacked.std():.2f}")
print(f"  Min: {weights_stacked.min():.2f}")
print(f"  Max: {weights_stacked.max():.2f}")

# Plot weight distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Weight distribution (log scale for better visualization)
axes[0].hist(np.log10(weights_stacked.values), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_title('Distribution of Weights (log10 scale)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('log10(Weight)', fontsize=10)
axes[0].set_ylabel('Frequency', fontsize=10)
axes[0].grid(True, alpha=0.3)

# Weight over time (median daily weight)
weights_daily_median = weights_stacked.groupby(level=0).median()
axes[1].plot(weights_daily_median.index, weights_daily_median.values, linewidth=1, alpha=0.7)
axes[1].set_title('Median Daily Weight Over Time', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Date', fontsize=10)
axes[1].set_ylabel('Median Weight', fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5.4 Calculate Weights from Residual Volatility

Estimate residual variance for each stock using rolling window, then calculate weights:
$$w_i = \frac{1}{\hat{\sigma}_i^2 + \epsilon}$$

In [ ]:
print("="*80)
print("ORGANIZING RESULTS")
print("="*80)

if len(factor_returns_list) > 0:
    # Factor returns: (n_dates, n_factors)
    # Each row = factor returns on that day
    print("\nCreating factor returns matrix...")
    df_factor_returns = pd.DataFrame(factor_returns_list)
    df_factor_returns.index.name = 'date'
    
    print(f"✓ Factor returns matrix created")
    print(f"  Shape: {df_factor_returns.shape}")
    print(f"  Dates: {len(df_factor_returns)}")
    print(f"  Factors: {len(df_factor_returns.columns)}")
    
    # Residuals: stack all daily residuals
    print("\nCreating residuals matrix...")
    df_residuals = pd.concat(residuals_list, axis=0)
    df_residuals.name = 'residual'
    
    print(f"✓ Residuals created")
    print(f"  Total residuals: {len(df_residuals):,}")
    
    # R-squared: (n_dates,)
    print("\nCreating R-squared series...")
    df_r_squared = pd.Series(r_squared_list, index=df_factor_returns.index, name='r_squared')
    
    print(f"✓ R-squared created")
    print(f"  Days: {len(df_r_squared)}")
    
    # Number of stocks per day
    df_n_stocks = pd.Series(n_stocks_list, index=df_factor_returns.index, name='n_stocks')
    
    print(f"\n✓ All results organized")
    
    # Display samples
    print("\nFactor returns (first 10 days, first 5 factors):")
    display(df_factor_returns.iloc[:10, :5])
    
    print("\nR-squared (first 10 days):")
    display(df_r_squared.head(10))
    
else:
    print("❌ No successful regressions!")
    df_factor_returns = pd.DataFrame()
    df_residuals = pd.Series()
    df_r_squared = pd.Series()
    df_n_stocks = pd.Series()

## 5.5 Regression Diagnostics

### Check Regression Quality:
1. **R-squared**: How much variance is explained by factors?
2. **Residuals**: Are they centered at zero with no trend?
3. **Factor returns**: Are they stable over time?

In [ ]:
print("="*80)
print("REGRESSION DIAGNOSTICS")
print("="*80)

if len(df_factor_returns) > 0:
    # R-squared statistics
    print("\n1. R-Squared Statistics:")
    print(f"   Mean: {df_r_squared.mean():.4f}")
    print(f"   Median: {df_r_squared.median():.4f}")
    print(f"   Std: {df_r_squared.std():.4f}")
    print(f"   Min: {df_r_squared.min():.4f}")
    print(f"   Max: {df_r_squared.max():.4f}")
    
    # Plot R-squared over time
    fig, axes = plt.subplots(2, 1, figsize=(14, 10))
    
    # R-squared time series
    axes[0].plot(df_r_squared.index, df_r_squared.values, linewidth=1, alpha=0.7)
    axes[0].axhline(df_r_squared.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {df_r_squared.mean():.3f}')
    axes[0].set_title('Daily R-Squared Over Time', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Date', fontsize=12)
    axes[0].set_ylabel('R-Squared', fontsize=12)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # R-squared distribution
    axes[1].hist(df_r_squared.values, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
    axes[1].axvline(df_r_squared.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {df_r_squared.mean():.3f}')
    axes[1].axvline(df_r_squared.median(), color='green', linestyle='--', linewidth=2, label=f'Median: {df_r_squared.median():.3f}')
    axes[1].set_title('R-Squared Distribution', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('R-Squared', fontsize=12)
    axes[1].set_ylabel('Frequency', fontsize=12)
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Residual statistics
    print("\n2. Residual Statistics:")
    print(f"   Mean: {df_residuals.mean():.8f} (should be ~0)")
    print(f"   Std: {df_residuals.std():.6f}")
    print(f"   Skewness: {df_residuals.skew():.4f}")
    print(f"   Kurtosis: {df_residuals.kurtosis():.4f}")
    
    # Plot residual distribution
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histogram
    axes[0].hist(df_residuals.values, bins=100, color='coral', edgecolor='black', alpha=0.7, range=(-0.1, 0.1))
    axes[0].axvline(0, color='red', linestyle='--', linewidth=2, label='Zero')
    axes[0].axvline(df_residuals.mean(), color='green', linestyle='--', linewidth=2, label=f'Mean: {df_residuals.mean():.6f}')
    axes[0].set_title('Residual Distribution (Clipped to [-0.1, 0.1])', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Residual', fontsize=12)
    axes[0].set_ylabel('Frequency', fontsize=12)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Q-Q plot
    stats.probplot(df_residuals.values, dist="norm", plot=axes[1])
    axes[1].set_title('Q-Q Plot: Residuals vs Normal Distribution', fontsize=14, fontweight='bold')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Number of stocks per day
    print("\n3. Number of Stocks Per Day:")
    print(f"   Mean: {df_n_stocks.mean():.0f}")
    print(f"   Median: {df_n_stocks.median():.0f}")
    print(f"   Min: {df_n_stocks.min():.0f}")
    print(f"   Max: {df_n_stocks.max():.0f}")
    
else:
    print("⚠ No results to diagnose")

## 5.6 Cumulative Factor Returns

Plot cumulative returns for top-performing factors.

In [ ]:
print("="*80)
print("CUMULATIVE FACTOR RETURNS")
print("="*80)

if len(df_factor_returns) > 0:
    # Calculate cumulative returns
    print("\nCalculating cumulative returns...")
    df_cumulative = (1 + df_factor_returns).cumprod()
    
    # Get top 10 factors by Sharpe ratio
    top_factors = factor_stats.head(10).index
    
    # Plot cumulative returns for top factors
    fig, ax = plt.subplots(figsize=(14, 7))
    
    for factor in top_factors:
        ax.plot(df_cumulative.index, df_cumulative[factor], label=factor, linewidth=1.5, alpha=0.8)
    
    ax.axhline(1, color='black', linestyle='--', linewidth=1, label='Baseline')
    ax.set_title('Cumulative Returns: Top 10 Factors by Sharpe Ratio', fontsize=14, fontweight='bold')
    ax.set_xlabel('Date', fontsize=12)
    ax.set_ylabel('Cumulative Return', fontsize=12)
    ax.legend(loc='best', fontsize=8, ncol=2)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print("\n✓ Cumulative returns plotted")
    
else:
    print("⚠ No factor returns to plot")

## 5.7 Weighted Least Squares with Heteroskedasticity Adjustment

### Formula (Equation 6.3):
$$\hat{\mathbf{f}}_t = (\mathbf{B}_t^T \boldsymbol{\Omega}_{\epsilon}^{-1} \mathbf{B}_t)^{-1} \mathbf{B}_t^T \boldsymbol{\Omega}_{\epsilon}^{-1} \mathbf{r}_t$$

Where:
- $\boldsymbol{\Omega}_{\epsilon} = \text{diag}(\sigma_1^2, \sigma_2^2, ..., \sigma_n^2)$: Residual covariance matrix (diagonal)
- $\sigma_i^2$: Estimated variance of stock $i$'s residuals

We estimate $\sigma_i^2$ using a rolling window of past residuals.

In [ ]:
print("="*80)
print("CROSS-SECTIONAL REGRESSION - SUMMARY REPORT")
print("="*80)

print(f"\nReport Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

print("\n" + "="*80)
print("REGRESSION SETUP")
print("="*80)

print(f"\nModel: r_t = B_t @ f_t + epsilon_t")
print(f"  r_t: returns vector (n_stocks,)")
print(f"  B_t: loadings matrix (n_stocks, n_factors)")
print(f"  f_t: factor returns (n_factors,) - ESTIMATED")
print(f"  epsilon_t: residuals (n_stocks,)")

print(f"\nEstimation Method: Weighted Least Squares (WLS)")
print(f"  Two-Pass Approach:")
print(f"    Pass 1: Unweighted regression → initial residuals")
print(f"    Weights: w_i = 1 / (rolling_std(residual_i)^2 + epsilon)")
print(f"    Pass 2: WLS regression with weights")
print(f"\n  Formula (Equation 6.3):")
print(f"    f_t = (B_t^T W_t B_t)^(-1) B_t^T W_t r_t")
print(f"\n  Weight estimation:")
print(f"    Rolling window: {ROLLING_WINDOW if 'ROLLING_WINDOW' in dir() else 60} days")
print(f"    Minimum periods: {MIN_PERIODS if 'MIN_PERIODS' in dir() else 20} days")

print("\n" + "="*80)
print("REGRESSION RESULTS")
print("="*80)

if len(df_factor_returns) > 0:
    print(f"\nDates processed: {len(df_factor_returns):,}")
    print(f"Date range: {df_factor_returns.index[0]} to {df_factor_returns.index[-1]}")
    print(f"Factors: {len(df_factor_returns.columns)}")
    
    print(f"\nR-Squared:")
    print(f"  Mean: {df_r_squared.mean():.4f}")
    print(f"  Median: {df_r_squared.median():.4f}")
    print(f"  Range: [{df_r_squared.min():.4f}, {df_r_squared.max():.4f}]")
    
    print(f"\nResiduals:")
    print(f"  Mean: {df_residuals.mean():.8f} (should be ~0)")
    print(f"  Std: {df_residuals.std():.6f}")
    
    print(f"\nFactor Returns:")
    print(f"  Mean Sharpe ratio: {factor_sharpe.mean():.4f}")
    print(f"  Best factor: {factor_sharpe.idxmax()} (Sharpe: {factor_sharpe.max():.4f})")
    print(f"  Worst factor: {factor_sharpe.idxmin()} (Sharpe: {factor_sharpe.min():.4f})")

print("\n" + "="*80)
print("OUTPUT FILES")
print("="*80)

if len(df_factor_returns) > 0:
    print(f"\nFactor returns: {factor_returns_path}")
    print(f"  Shape: {df_factor_returns.shape}")
    print(f"  Contains: Daily factor returns (WLS estimates)")
    
    print(f"\nResiduals: {residuals_path}")
    print(f"  Contains: Idiosyncratic returns (for Step 6)")
    
    print(f"\nDiagnostics: {diagnostics_path}")
    print(f"  Contains: R-squared, n_stocks per day")
    
    print(f"\nFactor statistics: {factor_stats_path}")
    print(f"  Contains: Mean, std, Sharpe ratio for each factor")

print("\n" + "="*80)
print("✅ STEP 5 COMPLETE: CROSS-SECTIONAL REGRESSION (WLS)")
print("="*80)
print("\nNext Step: Step 6 - Time-Series Estimation")
print("  Estimate factor covariance matrix and idiosyncratic risk")

## 5.8 Save Results

In [ ]:
print("="*80)
print("SAVING RESULTS")
print("="*80)

if len(df_factor_returns) > 0:
    # Save factor returns
    factor_returns_path = 'factor_returns_step5.parquet'
    print(f"\nSaving factor returns to {factor_returns_path}...")
    df_factor_returns.to_parquet(factor_returns_path, compression='snappy')
    
    import os
    file_size_mb = os.path.getsize(factor_returns_path) / 1024**2
    
    print(f"✓ Factor returns saved")
    print(f"  File: {factor_returns_path}")
    print(f"  Size: {file_size_mb:.2f} MB")
    print(f"  Shape: {df_factor_returns.shape}")
    
    # Save residuals
    residuals_path = 'residuals_step5.parquet'
    print(f"\nSaving residuals to {residuals_path}...")
    df_residuals.to_frame().to_parquet(residuals_path, compression='snappy')
    
    file_size_mb = os.path.getsize(residuals_path) / 1024**2
    print(f"✓ Residuals saved")
    print(f"  File: {residuals_path}")
    print(f"  Size: {file_size_mb:.2f} MB")
    
    # Save diagnostics
    diagnostics = pd.DataFrame({
        'r_squared': df_r_squared,
        'n_stocks': df_n_stocks
    })
    diagnostics_path = 'regression_diagnostics_step5.csv'
    diagnostics.to_csv(diagnostics_path)
    print(f"\n✓ Diagnostics saved to {diagnostics_path}")
    
    # Save factor statistics
    factor_stats_path = 'factor_statistics_step5.csv'
    factor_stats.to_csv(factor_stats_path)
    print(f"✓ Factor statistics saved to {factor_stats_path}")
    
else:
    print("❌ No results to save!")

## 5.10 Summary Report

In [ ]:
print("="*80)
print("CROSS-SECTIONAL REGRESSION - SUMMARY REPORT")
print("="*80)

print(f"\nReport Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

print("\n" + "="*80)
print("REGRESSION SETUP")
print("="*80)

print(f"\nModel: r_t = B_t @ f_t + epsilon_t")
print(f"  r_t: returns vector (n_stocks,)")
print(f"  B_t: loadings matrix (n_stocks, n_factors)")
print(f"  f_t: factor returns (n_factors,) - ESTIMATED")
print(f"  epsilon_t: residuals (n_stocks,)")

print(f"\nEstimation Method: OLS (Equation 6.4)")
print(f"  f_t = (B_t^T B_t)^(-1) B_t^T r_t")

print("\n" + "="*80)
print("REGRESSION RESULTS")
print("="*80)

if len(df_factor_returns) > 0:
    print(f"\nDates processed: {len(df_factor_returns):,}")
    print(f"Date range: {df_factor_returns.index[0]} to {df_factor_returns.index[-1]}")
    print(f"Factors: {len(df_factor_returns.columns)}")
    
    print(f"\nR-Squared:")
    print(f"  Mean: {df_r_squared.mean():.4f}")
    print(f"  Median: {df_r_squared.median():.4f}")
    print(f"  Range: [{df_r_squared.min():.4f}, {df_r_squared.max():.4f}]")
    
    print(f"\nResiduals:")
    print(f"  Mean: {df_residuals.mean():.8f} (should be ~0)")
    print(f"  Std: {df_residuals.std():.6f}")
    
    print(f"\nFactor Returns:")
    print(f"  Mean Sharpe ratio: {factor_sharpe.mean():.4f}")
    print(f"  Best factor: {factor_sharpe.idxmax()} (Sharpe: {factor_sharpe.max():.4f})")
    print(f"  Worst factor: {factor_sharpe.idxmin()} (Sharpe: {factor_sharpe.min():.4f})")

print("\n" + "="*80)
print("OUTPUT FILES")
print("="*80)

if len(df_factor_returns) > 0:
    print(f"\nFactor returns: {factor_returns_path}")
    print(f"  Shape: {df_factor_returns.shape}")
    print(f"  Contains: Daily factor returns for all factors")
    
    print(f"\nResiduals: {residuals_path}")
    print(f"  Contains: Idiosyncratic returns (for Step 6)")
    
    print(f"\nDiagnostics: {diagnostics_path}")
    print(f"  Contains: R-squared, n_stocks per day")
    
    print(f"\nFactor statistics: {factor_stats_path}")
    print(f"  Contains: Mean, std, Sharpe ratio for each factor")

print("\n" + "="*80)
print("✅ STEP 5 COMPLETE: CROSS-SECTIONAL REGRESSION")
print("="*80)
print("\nNext Step: Step 6 - Time-Series Estimation")
print("  Estimate factor covariance matrix and idiosyncratic risk")